Le but est de prédire si un client peu obtenir le crédit ou pas. 1= client à risque et 0=capable de rembourser
Dans la version streamlit je souhaite :
- choisir depuis un menu déroulant un ID client de la première colonne du dataframe "SK_ID_CURR" , une ligne du data frame.
- Faire la prédiction proba entre 0 et 1. Pourquoi pas avec une jauge à aiguille voir si elle atteint le seuil
- Avoir la prédiction 0 ou 1 en fonction du seuil 0=validé, 1 = refusé
- Avoir le shap des features importances pour comprendre le résultat


In [ ]:
import streamlit as st
import pandas as pd
import joblib
import shap
import numpy as np
import plotly.graph_objects as go
import matplotlib.pyplot as plt

In [ ]:
# Chargement
model = joblib.load("model.pkl")  # pipeline
threshold = joblib.load("threshold_lgbm.pkl")
data = pd.read_csv("app_data.csv")

In [1]:
# sélection du client
client_id = st.selectbox(
    "Choisir un client",
    data["SK_ID_CURR"]
)

client_data = data[data["SK_ID_CURR"] == client_id]


In [ ]:
# Uniquement les variables explicatives(ID et index ne sont pas des valeurs prédictives)
feats = [f for f in data.columns if f not in ['TARGET','SK_ID_CURR','SK_ID_BUREAU','SK_ID_PREV','index']]
X_client = client_data[feat]

# Prédiction en appliquant le seuil score métier
proba = model.predict_proba(X_client)[0, 1]
prediction = int(proba >= threshold)

In [ ]:
if prediction == 1:
    decision = "Crédit refusé"
else:
    decision = "Crédit accordé"

st.metric("Probabilité de défaut", f"{proba:.2f}")
st.write("Décision :", decision)

In [ ]:

# jauge de risque
fig = go.Figure(go.Indicator(
    mode="gauge+number",
    value=proba,
    title={'text': "Risque de défaut"},
    gauge={
        'axis': {'range': [0, 1]},
        'threshold': {
            'line': {'color': "red", 'width': 4},
            'value': threshold
        }
    }
))

st.plotly_chart(fig)

In [ ]:
# feature important
fig, ax = plt.subplots()
shap.plots.waterfall(shap_values[0], show=False)
st.pyplot(fig)